# Лекция 6. Чёрный ящик: трансферируемость и запрос-ограниченные атаки

Демонстрация: перенос атаки между моделями (transfer attack) и простая decision-based атака.

## 1. Обучение двух разных моделей (target и surrogate)

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
torch.manual_seed(0)
np.random.seed(0)

import torchvision
import torchvision.transforms as T

transform = T.Compose([T.ToTensor()])
train_ds = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_ds = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=1, shuffle=True)

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1,16,3,padding=1)
        self.conv2 = nn.Conv2d(16,32,3,padding=1)
        self.fc = nn.Linear(32*7*7,10)
    def forward(self,x):
        x = F.relu(self.conv1(x)); x = F.max_pool2d(x,2)
        x = F.relu(self.conv2(x)); x = F.max_pool2d(x,2)
        x = x.view(x.size(0),-1)
        return self.fc(x)

model = SimpleCNN()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
for i,(xb,yb) in enumerate(train_loader):
    opt.zero_grad(); loss = F.cross_entropy(model(xb), yb); loss.backward(); opt.step()
    if i>=300: break
print("Базовая модель обучена, loss:", loss.item())

class SmallMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28*28,128)
        self.fc2 = nn.Linear(128,10)
    def forward(self,x):
        x = x.view(x.size(0),-1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

surrogate = SmallMLP()
opt2 = torch.optim.Adam(surrogate.parameters(), lr=1e-3)
for i,(xb,yb) in enumerate(train_loader):
    opt2.zero_grad(); loss2 = F.cross_entropy(surrogate(xb), yb); loss2.backward(); opt2.step()
    if i>=300: break
print("Surrogate MLP обучена, loss:", loss2.item())


Базовая модель обучена, loss: 0.11312925815582275
Surrogate MLP обучена, loss: 0.3214459717273712


## 2. Transfer attack: атака строится на surrogate, применяется к target CNN

In [2]:

def fgsm_attack(model, x, y, eps):
    x = x.clone().detach().requires_grad_(True)
    loss = F.cross_entropy(model(x), y)
    loss.backward()
    return torch.clamp(x + eps*x.grad.sign(), 0, 1).detach()

n_test = 300
transfer_success = 0
for i, (x,y) in enumerate(test_loader):
    if i>=n_test: break
    x_adv = fgsm_attack(surrogate, x, y, eps=0.25)
    pred_target = model(x_adv).argmax(1)
    if pred_target.item() != y.item():
        transfer_success += 1
print(f"Trансферируемость атаки surrogate->target CNN: ASR={transfer_success/n_test:.2%}")


Trансферируемость атаки surrogate->target CNN: ASR=53.33%


## 3. Простая decision-based атака (случайный поиск по границе)

In [3]:

def simple_decision_attack(model, x, y, steps=200, step_size=0.05):
    x_adv = torch.rand_like(x)
    for _ in range(steps):
        if model(x_adv).argmax(1).item() != y.item():
            direction = (x - x_adv)
            x_adv = x_adv + step_size * direction / (direction.norm()+1e-9)
            x_adv = torch.clamp(x_adv, 0, 1)
        else:
            x_adv = x_adv + step_size*torch.randn_like(x_adv)
            x_adv = torch.clamp(x_adv, 0, 1)
    return x_adv.detach()

x, y = next(iter(test_loader))
x_db = simple_decision_attack(model, x, y)
print("Истинный класс:", y.item(), "Предсказание после decision-based атаки:", model(x_db).argmax(1).item())
print("Использован только доступ к argmax-решению модели, без градиентов и вероятностей")


Истинный класс: 5 Предсказание после decision-based атаки: 1
Использован только доступ к argmax-решению модели, без градиентов и вероятностей
